# ImageDETECTIVE: fakes from Kandinsky 2.2 (Kaggle)

A third generator for training, with its own decoder (MoVQ), from all 792 pair captions in the repo.
All logic is in `scripts/generate_open_models.py`. Kandinsky 2.2 is Apache 2.0 and open: no token needed.

**Settings before running** (right panel): *Accelerator*: **GPU T4 x2** (one GPU is used), *Internet*: **on**.

**Two steps:** first run all cells with `LIMIT = 3` (quick test). If the images look right, set `LIMIT = None`,
then **Save Version → Save & Run All (Commit)**. The images end up in the version's output: `open_gen/k22/`.

In [ ]:
LIMIT = 3  # quick test: 3 images. Full run: None

In [ ]:
import os

os.environ["HF_HOME"] = "/tmp/hf"  # model downloads stay out of the notebook output
!rm -rf /tmp/repo && git clone --depth 1 https://github.com/phantom007xy/ImageDETECTIVE.git /tmp/repo
# Kaggle's preinstalled torchao is too old for diffusers 0.41 (no FqnToConfig) and crashes its import;
# we use no quantization, so without torchao diffusers simply skips that part.
!pip uninstall -y -q torchao
!pip install -q diffusers==0.41.0
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
import subprocess
import sys

OUT = "/kaggle/working/open_gen"
os.makedirs(OUT, exist_ok=True)
cmd = [sys.executable, "scripts/generate_open_models.py", "--model", "k22", "--prompts",
       "data/prompts/pairs_v1_prompts.csv", "data/prompts/pairs_v2_prompts.csv", "--out-dir", OUT, "--device", "cuda:0"]
if LIMIT is not None:
    cmd += ["--limit", str(LIMIT)]
with open(f"{OUT}/k22.log", "w") as log:
    code = subprocess.run(cmd, cwd="/tmp/repo", stdout=log, stderr=subprocess.STDOUT).returncode
print("exit code:", code)
!tail -n 8 {OUT}/k22.log

In [ ]:
import csv
import glob
from collections import Counter

from PIL import Image

path = f"{OUT}/k22/manifest.csv"
rows = list(csv.DictReader(open(path))) if os.path.exists(path) else []
seconds = [float(r["seconds"]) for r in rows]
print("k22", dict(Counter(r["status"] for r in rows)),
      f"| {sum(seconds) / max(1, len(seconds)):.1f} s per image" if seconds else "")
for p in sorted(glob.glob(f"{OUT}/k22/*.png"))[:3]:  # a few images to look at
    im = Image.open(p)
    display(im.resize((384, round(384 * im.height / im.width))))